<a href="https://colab.research.google.com/github/Living-with-machines/dhoxss-text2tech/blob/hf/Sessions/5e_LLMs_as_Research_Assistants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using (open-source) LLMs for analysing humanities data

In this notebook, we explore some applications of generative AI to historical newspaper data.

## LLMs as Research Assistants

The overarching questions is: how to merge (information in) historical data with the predictive abilities of language models?

**Language Modelling:** the previous notebook investigated how language models **'absorb'** historical knowledge via continued pre-training on historical data using a language modelling task.

**Instruction Following:** What sets the current generation of language models (ChatGPT, Llama, Claude, etc.) apart from the previous models is their ability to follow instructions, usually based on a given prompt.
- **Few-shot learning/In-context learning**: "I tell you (i.e. the LLM) what to do based on giving a few examples of correct answers."
- **Retrieval Augmented Generation** (RAG): "I fetch a few documents, on which you have to base your answer"

This notebook explores the latter approach, where LLMs are used to "analyse" content in meaningful and often complex ways.

## Why work with historical newspapers?

[Example](https://www.britishnewspaperarchive.co.uk/viewer/BL/0000080/18380701/004/0001?browse=true)

- **Big**: Immense collections of data (if not the largest).
- **Fine-grained**: often daily reports on a myriad of events, from the 'banal' to the 'breaking' news.
- **Longitudinal**: newspapers run for decades (even though their content and formats change over time.

### Case Study: Accidents in the News
#### Or: Using Large Language Models to Investigate "Small" People

![](https://global.oup.com/academic/covers/pop-up/9780198732334)

Inspired by Paul Fyfe's book: ["By Accident or Design"](https://global.oup.com/academic/product/by-accident-or-design-9780198732334?cc=us&lang=en&)

- We treat newspapers as periodical purveyors of miscellaneous content: “A newspaper without an account of one or more accidents . . . is scarcely ever taken up”
- Accident reports are “subjective attempts at the construction of evidence” (Roger Cooter) and “useful as indices to social and especially industrial change” (Roger Lane).

## 'Baby' RAG (Or elevated copy-pasting)

The "I tell you what to do" is usually referred to a "prompt" which we ask the LM to "complete" (sometimes using few-shot learning).

**Q: How can we inject historical data in this process?**

**A: Retrieval Augmented Generation**: We ask the LLM to answer questions based on historical content that we "copy-paste" into our prompt.

- Given a query, something we want to know
- We select documents from our database that might contain the answer to this question
- Then we use the language model to generate an answer based on the documents we retrieved.

We focus on the "generation" less on the "retrieval". We want to keep things simple (at the technical level)!

## Open Source LLMs


We will be largely playing with an open-source model, **Llama-3-Instruct**, to get a feeling of how LLMs change the way we can interrogate historical data.

Many of the popular, commercial models are "closed" in the sense that even though you can interact with them, they remain "hidden", i.e. you can not "download" them or freely run and manipulate them on your own computer system.


### Why Open-source?

- **Privacy:**: You might not want to share your data (and ideas) with companies such as OpenAI;
- **Cost:** Making abstraction of the caveat above, using open-source models might reduce costs if you want to apply for example a prompt to 10k newspaper articles;
- **Transparency:** Be mindful that there are different gradations of openness and transparency. Even when you can access the model weights, you might remain in the dark about training data and other factors);
- **Flexibility:** Although some providers allow you to train or fine-tune closed models on your data (ties in with privacy), open-source models still give you more freedom and wiggle room to build new models and applications.

### Technical note

We will rely on the Hugging Face `InferenceClient` to access LLMs. These are freely accessible, but rate limits apply! If you would want to deploy a 'local' version (we're still on Colab, but the code should also work on your computer), uncomment the code below (where indicated) and make sure you are using a [GPU](https://cloud.google.com/gpu). To select a GPU on Colab, Go to **`Runtime`** and select **`Change runtime type`**, then select `T4 GPU` (or any other GPU available).



This notebook is inspired by: https://huggingface.co/learn/cookbook/structured_generation

## The Hugging Face Hub

In the examples below, we will experiment with `Llama-3-8B-Instruct`, a recent series of open-source LLMs created by Meta. To use Llama3, you need to:

- Make an account on Hugging Face: https://huggingface.co/
- Go to the Llama-3-8B and sign the terms of use you should get a reply swiftly: https://huggingface.co/meta-llama/Meta-Llama-3-8B
- Create a user access token with at least read access: https://huggingface.co/docs/hub/en/security-tokens
- Run the code cell below to log into the Hugging Face hub. Copy-paste the access token.
- Reply `n` to the question 'Add token as git credential? (Y/n)'

In [ ]:
!hf auth login

## Preparing model and data

### Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore') # disable warnings

In [ ]:
import transformers
from huggingface_hub import InferenceClient
#from datasets import Dataset
from tqdm import tqdm
import pandas as pd
import torch
import pandas as pd
import json
pd.set_option("display.max_colwidth", 100)

### Load model

In [ ]:
# choose a LLMs model
repo_id = "meta-llama/Meta-Llama-3-8B-Instruct"
# instantiate the inference client
llm_client = InferenceClient(model=repo_id, timeout=120)

In [ ]:
# use this cell if you can access an A100 or L4 GPU
# define the model, we use the instruct variant
checkpoint = "meta-llama/Meta-Llama-3-8B-Instruct"
device = 'cuda' # make sure you use a GPU

# instantiate a text generation pipeline
pipeline = transformers.pipeline(
    "text-generation",
    model=checkpoint,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda",
)

# some fluff to improve the generation
terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [ ]:
# # use this cell if you can only access a T4 GPU
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
# # define the model, we use the instruct variant
# checkpoint = "meta-llama/Meta-Llama-3-8B-Instruct"
# device = 'cuda' # make sure you use a GPU if available

# bnb_confic = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# #tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(checkpoint,
#                                              quantization_config=bnb_confic,
#                                              device_map='auto')

# pipeline = transformers.pipeline(
#     "text-generation",
#     model=model,
#     tokenizer= tokenizer,
# )


# # some fluff to improve the generation
# terminators = [
#     pipeline.tokenizer.eos_token_id,
#     pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
# ]

## Prompting

LLM generate text from an input, usually referred to as a 'prompt', a piece of text we like the model to use as a starting point for predicting novel tokens.

When 'chatting' with an LLM, we usually provide the model with (at least) two messages: a system and a user prompt or message.

**System message**:

- **Generic instructions on behaviour**: specify how the model should behave (e.g. be helpful, respectful, neutral) or the role it should play (e.g., a teacher, assistant, or advisor).
- **Constraints**: Specific instructions on what the model should avoid or how it should generate responses.
- **Context**: Background information or context that remains constant throughout the session to ensure consistency.

**User message**:

- **Query**: specifies input from the user, such as a question, instruction, or request that the model needs to respond to.
- **Dynamic**: changes with each interaction, reflecting the user's immediate needs, questions, or instructions.

The Hugging Face chat prompt template allows messages as lists of dictionaries.

```python
messages [
 {
    "role" : "system",
    "content": "<system prompt here>"
 },
 {
    "role" : "user",
    "content": "<user prompt here>"
 }
]
```

## RAG by hand

Define a message by articulating a system and user prompt.

In [ ]:
messages = [
    {
        "role": "system",
        "content": """
          You are a helpful AI that will assist me with analysing and reading newspaper articles.
          Read the newspaper article attentively and provide a short description of principal characters.
          The newspaper article is enclosed within triple hashtags (i.e. ###).
          Don't make things up! If the information is not in the article then reply 'I don't know'
          """
              },

    {
        "role": "user",
        "content": f"""
                  ###POOR T,i,ENIPAT A 1„k CT  The Poor Law Coirdnissioti(rs have issued a ei; cular,
                  dated the 20th instant, stating that they have consulted the Attorney and
                  Solicitor-General on the construction of the late Removal Act, and give as the
                  result:— I. " That the proviso to the Ist section of the 9 and 10 Vict., c. 66,
                  which sets forth the exceptions to the principal enactments that are to be
                  excluded in the computation of time, is net retrospective in its operation, so
                  as to apply to cases where the five years\' residence was complete before the statute.
                  2. " That an interval between the completion of the five years residence and the
                  application for the warrant of removal filled up by one of the exceptions contained
                  in the proviso will not p event the operation of the statute in restraining the
                  removal of the pauper whu had resided for the specified time. 3. " That orders
                  of removal obtained previous to th• passing of the Act, but not then executed
                  by the removal of the paupers,###"""
              }
  ]

In [ ]:
messages

In [ ]:
#help(llm_client.chat_completion)

In [ ]:
# uncomment this code if you want to work locally, comment the other function
def get_completion(messages: list, temperature=.1, top_p=.1) -> str:
  """get completion for given system and user prompt
    Arguments:
    messages (list): a list containin a system and user message as
      python dictionaries with keys 'role' and 'content'
    temperature (float): regulate creativity of the text generation
    top_p (float): cummulative probability included in the
      generation process
  """
  prompt = pipeline.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
      )

  outputs = pipeline(
    prompt,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=temperature,
    top_p=top_p,
      )
  return outputs[0]["generated_text"][len(prompt):]

# uncomment this if you are using the llm_client
# def get_completion(messages: list, temperature=.0, top_p=.1,grammar=None):
#     """get completion for given system and user prompt
#       Arguments:
#         messages (list): a list containin a system and user message as
#           python dictionaries with keys 'role' and 'content'
#         temperature (float): regulate creativity of the text generation
#         top_p (float): cummulative probability included in the
#           generation process
#     """
#     outputs = llm_client.chat_completion(
#         messages=messages,
#         max_tokens=1024,
#         temperature=temperature,
#         top_p=top_p
#         )
#     return outputs.choices[0].message.content

In [ ]:
print(get_completion(messages))

## Exercise

- Change the system message and ask the model to perform another taks, for example translate to pirate English, summarize the text or something silly!
- Change the user message, for inspiration you can go to the British Newspaper Archive, go to ["Advanced Search"](https://www.britishnewspaperarchive.co.uk/search/advanced) and select "Free To View"

In [ ]:
messages = [
    {"role": "system", "content": """ENTER PROMPT HERE"""
          },
    {"role": "user", "content": f"""###ENTER CONTENT HERE###"""}
]

print(get_completion(messages))


### Download data

We will be experimenting with a small set of 10k British newspaper articles provided by the ["Living with Machines"](https://livingwithmachines.ac.uk/public-domain-newspaper-titles-in-living-with-machines/) project.

In [ ]:
# download a sample of 10.000 newspaper articles
!gdown 1cewugDdehGn-wPP9B4kTBg_Wmu-14aHq
# unzip the downloaded sample
!unzip -o 0002247.csv.zip
!rm -r __MACOSX

In [ ]:
# df = pd.read_csv('https://raw.githubusercontent.com/kasparvonbeelen/uga-llm-workshop/refs/heads/main/newspapers/0002247.csv',index_col=0)
# df.head(3)

In [ ]:
df = pd.read_csv('0002247.csv')
df.head(3)

In [ ]:
df.shape

### Issues with the data

- Segmentation: ["what is an article"](https://docs.google.com/document/d/1fbfWaDx6P-VV09j7pC__Rez_WVJOnnhq4RS-yiQ-XBM/edit?usp=sharing)
- OCR Quality: 6ibb*riSH?

In [ ]:
print(df.iloc[0].content)

In [ ]:
print(df.iloc[10].content)

### Process data

To facilitate the analysis, we divide the newspaper articles into smaller, hopefully more meaningful chunks of 250 words (with a 50-word overlap).

- We split by proxy article (i.e. double hard returns)
- Remove single hard returns from the proxy articles.

In [ ]:
def get_chunks(text: str, size: int=250,step: int=50) -> list:
  """divide a text into chunks of similar size
  Arguments:
    text (str): input text
    size (int): number of tokens in each chunk
    step (int): step size
  Returns a list of strings
  """
  words = text.split()
  return [' '.join(words[i:i+size]) for i in range(0,len(words),step)]

In [ ]:
# split by proxy-article
df['elements'] = df.content.apply(lambda x: [' '.join(ch.split('\n')) for ch in x.split('\n\n')])
# reorder the dataframe
# with one chunk in each row
# instead of the whole text
df_by_element = df.explode('elements')
# # apply chunking to text
df_by_element['chunks'] = df_by_element.elements.apply(lambda x: get_chunks(x))
df_chunks = df_by_element.explode('chunks')
df_chunks.reset_index(drop=True, inplace=True)
df_chunks.shape # wow that's a lot of chunks ;-)

In [ ]:
len(df_by_element.iloc[110].elements),len(df_by_element.iloc[110].chunks)

In [ ]:
df_chunks.iloc[5].chunks

## Retrieving articles about accidents

In [ ]:
import re
pattern = re.compile(r'\baccidents?\b', re.I) # compile a regex
pattern.findall('accidents accident AccIdent accidental') # test the regex on a few example


In [ ]:
tqdm.pandas()
df_chunks['chunk_count'] = df_chunks.progress_apply(lambda x: len(pattern.findall(str(x['chunks']))), axis=1)
df_chunks['title_count'] = df_chunks.progress_apply(lambda x: len(pattern.findall(str(x['title']))), axis=1)

In [ ]:
df_chunks.sort_values('chunk_count', ascending=False)[['title','chunk_count','title_count','chunks']][:10]

## Applying text generation to historical documents


### Step 1: Classify (and Clean)
#### Selecting and processing newspaper data

- Not all snippets (or chunks) that mention accidents are necessarily relevant, i.e. "about accidents".
- We can use Llama 3 to **classify** (and clean the data) using "few-shot learning".

Let's select ten chunks that mention 'accidents' most often (by way of example). Running the models can take quite some otherwise.

In [ ]:
df_chunks.chunk_count.value_counts(normalize=False)

In [ ]:
df_accident = df_chunks.sort_values('chunk_count',ascending=False)[:10]
df_accident[['title','chunk_count','title_count','chunks']]

Run the cell below to load the `apply_completions` function.

In [ ]:
def apply_completions(item: pd.Series,
                      system_message: str,
                      user_message: str='',
                      text_column: str = 'chunks') -> str:
  """
  Function that appl
  Argument:
    item (pd.Series): row from a pandas Dataframe
    system_message (str): system prompt, specifies how the system
      should behave in
    user_message (str): user prompt, give instruction how to
      process each historical. the documents itself will be append
      from the 'text_column' argument
    text_column (str): name of the text column
  """
  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
      ]
  messages[1]['content'] += f"\n\n###{item[text_column]}###"
  return  get_completion(messages)

We apply the prompt to the text chunks in our dataframe.

In [ ]:
tqdm.pandas() # use tqdm to view progress

system_message = """You are a document classifier that will clean and correct
    snippets of newspaper articles  as either about accidents or not about accidents.
    You answer with only 'yes', 'no', or 'unsure'.

    Examples are:
    Input: RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in Eng- land, which is published annually,
    Output: yes

    Input: LATEST INTELL IGEN CE. the prisoner was committed for trial for embezzlement. He was also further committed in two
    Output: no

    """

df_accident['classification'] =  df_accident.progress_apply(apply_completions,system_message=system_message, axis=1)

In [ ]:
df_accident['classification']

In [ ]:
#print the summaries
df_accident['classification'].value_counts()

In [ ]:
df_accident[df_accident["classification"]== 'no'].chunks.values

In [ ]:
tqdm.pandas() # use tqdm to view progress

system_message = """You are a document will clean and correct snippets of newspaper articles that mention the word 'accident'.
    Remove all text that is not about accidents, i.e. mention other events or facts that are not about accidents.
    Try to correct errors in the text wherever possible.
    Examples are:
    Input: "RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in Eng- land, which is published annually,
            POOR T,i,ENIPAT A 1„k CT  The Poor Law Coirdnissioti(rs have issued a ei; cular, dated the 20th instant, stating that they have consulted the Attorney and"
    Output: "RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in England, which is published annually."

    Input: "LATEST INTELL IGEN CE. the prisoner was committed for trial for embezzlement. He was also further committed in two"
    Output: ""
    """

df_accident['clean'] =  df_accident.progress_apply(apply_completions,system_message=system_message, axis=1)

In [ ]:
df_accident['clean']

We can remove the annoying boilerplate text such as "Here is the cleaned and corrected text:\n\n".

In [ ]:
df_accident['clean_article'] = df_accident['clean'].apply(lambda x: x.split(':\n\n')[-1].strip('#'))
df_accident['clean_article']

Another useful example is **summarisation.**

In [ ]:
tqdm.pandas()

system_message = """You are a helpful AI that will help analysing newspaper articles that mention the word 'accident'.
    Provide a short summary about the accident mentioned in the article, focussing on places, people and machines mentioned.
    """

df_accident['summary'] =  df_accident.progress_apply(
                                  apply_completions,
                                  system_message=system_message,
                                  text_column='clean_article',
                                  axis=1
                                )

In [ ]:
df_accident['summary_clean'] = df_accident['summary'].apply(lambda x: x.split(':\n\n')[-1])

In [ ]:
print(df_accident['clean'].iloc[9])

In [ ]:
print(df_accident['summary_clean'].iloc[9])

### Step 2: Structuring Information

The previous step helps us **prepare** data but not with the **analysis**.

One problem with RAG—as we introduced it so far—is that it remains **document-focused**, i.e. it extracts information from a sample of texts, which it uses to generate an answer.

**RAG is dominated by a Q&A paradigm which might be less useful for digital historians.** There are a few issues to discuss if we want to make RAG more useful for computational and historical analysis.

**Scale**: How can the "LLM as Research Assistant" paradigm help us with collections at a larger scale? For example, address questions such as: How did accidents in the news change over time? Who is involved in or blamed for the accidents? What are the locations where accidents take place? We could copy-paste all the articles in one prompt, but that is unlikely to work for multiple reasons.

**Text-in-text-out**: In one sense, using LLMs displaces (but does not solve) the problem of analysing text at a large scale. We simply move from input to output text, from article context to completions (correction, summaries, etc.)—except in the classification example, but this is not novel.

One solution I want to demonstrate in this workshop is the use of **structured completions**, using LLMs to go from unstructured text to structured information, which is more amenable to (quantitative) analysis.

Let's make this clear with a concrete example!


#### Structured Generation

Working with verbose and unstructured responses is difficult. Fortunately, we can ask the LLM to respond in a **structured fashion** that we can process more easily.

Let's have a look at how to extract structured **biographical information** from newspaper articles.

History is (often but not exclusively) about **people**.

Newspapers contain a lot of biographical information. In accident reports, we do get some background information about the people involved, implicitly (gender) or explicitly (professions or age). **Who are the victims involved in these news reports?** What stories can we tell based on these reports?

Below, we use a language model to extract biographical information from newspaper reports and return it in a specified, structured format, which allows us to analyse accidents reports as structured data.

### Writing a system prompt for structured generation

We rewrite the system prompt and give it a few more instructions on how to respond to our queries.
- We explain the **task** in natural language
- We make explicit the **structure** we expect

In [ ]:
system_message = """
    You are an helpful AI for analysing historical newspaper articles.
    Extract biographical information from a newspaper article between ###
    Return the biographical as a list of Python dictionaries.
    The information MUST be extracted from the newspaper articles, with spelling and wording identical to the source.
    This list of dictionaries should begin with a "START" tag and end with a "END" tag.
    Don't make thigs up! If you don't know the answer, simply return an empty list, i.e. [].


    Input:
    ###J. D. McPhill, a miner aged 58, died tragically in a railway accident, in which his wife M. M. McBilly, age 59, was also injured.###

    Output:
    START
    [
        {
          "name" : "J. D. McPhill",
          "gender" : "male",
          "profession" : "miner",
          "age" : 58,
          "outcome": "died",
          "cause": "railway accident"

        },
        {
          "name" : "M. M. McBilly",
          "gender" : "female",
          "age" : 58,
          "outcome": "injured",
          "cause": "railway accident"
        }

    ]
    END

    Input:
    """



In [ ]:
# input
snippet = df_chunks.sort_values('chunk_count',ascending=False).iloc[100].chunks; snippet

In [ ]:
# output
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": f'\n\n###{snippet}###'}
      ]

completion = get_completion(messages)
print(completion)

Let's now apply this prompt to a larger and different set of newspaper articles.

In [ ]:
df_small = df_chunks.sort_values('chunk_count',ascending=False)[100:110]
df_small

In [ ]:

df_small['bio'] =  df_small.progress_apply(apply_completions,system_message=system_message,text_column='chunks' , axis=1)

In [ ]:
#df_small = pd.read_json('https://raw.githubusercontent.com/kasparvonbeelen/uga-llm-workshop/refs/heads/main/newspapers/articles_with_bio.json')

In order to convert the response from the LLM (which is just a text) to a structured Python object, we need to write a small additional function `eval_completion`.

In [ ]:
# load a function for parison the output
from typing import List, Dict

def eval_completion(completion: str) -> List[Dict]:
  """Convert the completion as string to a Python list of dictionaries
  Argument:
      completion (str): structured generation by LLM
  """
  try:
    return eval(completion.split('START')[-1].strip().rstrip('END').strip())
  except Exception as e:
    #print(e)
    return []

In [ ]:
idx = 9547
print('Original output as string:')
print(df_small['bio'].loc[idx])
print()
print('_'*20)
print()
print('Formatted as Python object:')
eval_completion(df_small['bio'].loc[idx])

In [ ]:
# convert all the completions to Python objects
df_small['bio_structured']= df_small['bio'].apply(lambda x: eval_completion(x))

In [ ]:
# print(df_small[['chunks']].iloc[-2].values)
# print(df_small[['bio_structured']].iloc[-2].values)

Now we can convert the LLM outputs to a tabular format, which would be an excellent starting point for our historical analysis. Of course, the data requires further processing, but this is for another time. ;-)

In [ ]:
df_bios = pd.DataFrame([d for i,row in df_small.iterrows() for d in row['bio_structured']])
df_bios

In [ ]:
df_bios.age.value_counts().plot(kind='bar')

If you want to learn more about improving structured generation, I can recommend [this tutorial](https://huggingface.co/learn/cookbook/en/structured_generation).

Also, researchers are currently investigating to what extent forcing  LLMs to generate information in a specified format, lowers the quality of the outputs.

#### Question: How would you improve the prompt?

### GraphRAG

A recent topic of interest is [GraphRAG](https://arxiv.org/abs/2404.16130).

To make sense of large set of documents, we convert the corpus to a network of information. For example, we will convert statements "I got to home" to (subject, predicate, object) triples such as ("I", "go", "home"). Then we detect clusters in the network which we can  use to generate more answers to complex queries such as "what was the age of people involved in bike-accidents" etc.

There is more to this. But here. I'd just like to focus on the first part, the conversion of content to triples.

In [ ]:
tqdm.pandas() # use tqdm to view progress

system_message = """You are a document will clean and correct snippets of newspaper articles that mention the word 'accident'.
    Remove all text that is not about accidents, i.e. mention other events or facts that are not about accidents.
    Try to correct errors in the text wherever possible.
    Examples are:
    Input: "RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in Eng- land, which is published annually,
            POOR T,i,ENIPAT A 1„k CT  The Poor Law Coirdnissioti(rs have issued a ei; cular, dated the 20th instant, stating that they have consulted the Attorney and"
    Output: "RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in England, which is published annually."

    Input: "LATEST INTELL IGEN CE. the prisoner was committed for trial for embezzlement. He was also further committed in two"
    Output: ""

    """

df_small['clean'] =  df_small.progress_apply(apply_completions,system_message=system_message, axis=1)

In [ ]:
#df_small = pd.read_json('https://raw.githubusercontent.com/kasparvonbeelen/uga-llm-workshop/refs/heads/main/newspapers/articles_with_graph.json')

Let's therefore clean our data first.

In [ ]:
df_small['clean_article'] = df_small['clean'].apply(lambda x: x.split(':\n\n')[-1].strip('#'))
df_small['clean_article']

And then write a new system prompt. In this case, use one provided by [this excellent tutorial](https://towardsdatascience.com/how-to-convert-any-text-into-a-graph-of-concepts-110844f22a1a).

In [ ]:
# prompt borrowed from https://towardsdatascience.com/how-to-convert-any-text-into-a-graph-of-concepts-110844f22a1a
system_message = """You are a network graph maker who extracts terms and their relations from a given context. "
    "You are provided with a context chunk (delimited by ```) Your task is to extract the ontology "
    "of terms mentioned in the given context. These terms should represent the key concepts as per the context. \n"
    "Thought 1: While traversing through each sentence, Think about the key terms mentioned in it.\n"
        "\tTerms may include object, entity, location, organization, person, \n"
        "\tcondition, acronym, documents, service, concept, etc.\n"
        "\tTerms should be as atomistic as possible\n\n"
    "Thought 2: Think about how these terms can have one on one relation with other terms.\n"
        "\tTerms that are mentioned in the same sentence or the same paragraph are typically related to each other.\n"
        "\tTerms can be related to many other terms\n\n"
    "Thought 3: Find out the relation between each such related pair of terms. \n\n"
    "Format your output as a list of json. Each element of the list contains a pair of terms"
    "and the relation between them, like the follwing: \n"
    "```[\n"
    "   {\n"
    '       "node_1": "A concept from extracted ontology",\n'
    '       "node_2": "A related concept from extracted ontology",\n'
    '       "edge": "relationship between the two concepts, node_1 and node_2"\n'
    "   }, {...}\n"
    "]```"
    """

In [ ]:
df_small['graph'] =  df_small.progress_apply(apply_completions,system_message=system_message, text_column='clean_article',  axis=1)

Run the cell below to look at some examples.

In [ ]:
df_small['graph'][:3]

In [ ]:
def eval_completion_graph(completion: str):
  try:
     return eval(completion.split('```')[1].strip())
  except Exception as e:
    #print(e,completion)
    return []

df_small['json'] = df_small['graph'].apply(eval_completion_graph)

In [ ]:
df_small['json']

In [ ]:
knowledge_graph = []
for g in df_small['json'].to_list():
      for e in g:
        knowledge_graph.append((e['node_1'],e['edge'],e['node_2']))
graph_df = pd.DataFrame(knowledge_graph, columns=['node1','relation','node2'])
graph_df

In [ ]:
graph_df['relation'].value_counts()[:20]

In [ ]:
# Create directed graph
import networkx as nx
import matplotlib.pyplot as plt
G = nx.DiGraph()
triples = []
for node1, relation, node2 in knowledge_graph:
  #if "location" in relation:
      triples.append((node1, relation, node2))
      G.add_edge(node1, node2, label=relation)

# Plot the graph
plt.figure(figsize=(25, 25), dpi=300)
#pos = nx.spring_layout(G, k=2, iterations=10, seed=0)
pos = nx.kamada_kawai_layout(G,scale=3)

nx.draw_networkx_nodes(G, pos, node_size=500)
nx.draw_networkx_edges(G, pos, edge_color='gray', edgelist=G.edges(), width=2)
nx.draw_networkx_labels(G, pos, font_size=25)
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=15)

# Display the plot
plt.axis('off')
plt.show()

## Exercise

Experiment with your own system message! Have fun :-)

In [ ]:
# enter code here

# What if things don't work?

- Use larger models (see example below for using the OpenAI API)
- Model fine-tuning on real or synthetic data. An example [here](https://huggingface.co/blog/mlabonne/sft-llama3)

In [ ]:
!pip install openai

In [ ]:
import openai

In [ ]:
df_chunks.iloc[4]['chunks']

In [ ]:
from openai import OpenAI
client = OpenAI(api_key='sk-...')

In [ ]:
completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": "You are a helpful assistant. Correct the text below."},
    {"role": "user", "content": df_chunks.iloc[4]['chunks']}
  ]
)



In [ ]:
print(completion.choices[0].message.content)

# Fin.